# Module 02 Homework - Pandas
Using a dataset from _Wine Spectator_, a wine magazine, we will practice the material that we covered in notebook M02_Colab04 to M02_Colab06. This notebook will cover data transformation, grouping, and sorting using pandas.

Updated by Stephani (Seven) Marie Soriano (018072470)  
Last updated: 02/22/26

In [ ]:
# Optional: run %pip install pandas if needed

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 19.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 19.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]
Note: you may need to restart the kernel to use updated packages.


In [9]:
csvurl="https://gist.githubusercontent.com/clairehq/" + \
        "79acab35be50eaf1c383948ed3fd1129/raw/407a02139ae1e134992b90b4b2b8c329b3d73a6a/winemag-data-130k-v2.csv"
import pandas as pd
wine = pd.read_csv(csvurl)
wine.drop(columns=wine.columns[0], inplace=True)

**Data cleaning**  
Notice that the first column is redundant. Part of data analysis is cleaning and removing redundancy. How would you drop the redundant column inplace, that is overwrite the dataframe.

#### Question 1: ####  
What is the mean of the points column?

In [10]:
wine['points'].mean()

np.float64(88.43403716087269)

#### Question 2: ####  
How many countries are present in this dataset? (Only count each country once)

In [ ]:
wine['country'].nunique()

41

#### Question 3: ####
How many times does each country appeared in this dataset? Show each country and the corresponding count (show counts in ascending order)

In [12]:
wine['country'].value_counts().sort_values()

country
Slovakia                      1
Bosnia and Herzegovina        1
Armenia                       1
Switzerland                   4
India                         4
Luxembourg                    4
Ukraine                       5
Macedonia                     6
Cyprus                        6
Czech Republic                6
Serbia                        7
Peru                          8
Morocco                      11
Lebanon                      20
Moldova                      30
Brazil                       31
Mexico                       31
England                      36
Georgia                      37
Slovenia                     39
Turkey                       43
Croatia                      44
Uruguay                      61
Hungary                      61
Romania                      67
Bulgaria                     68
Canada                      108
Greece                      242
Israel                      259
New Zealand                 733
South Africa                737


#### Question 4: ####
Create a variable `adjusted_price` containing the adjusted price which is the price subtracted by the average price. *This is called **"centering" transformation** - a method commonly used in the preprocessing step before applying various machine learning algorithms.*

In [13]:
wine['adjusted_price'] = wine['price'] - wine['price'].mean()
adjusted_price = wine['adjusted_price']

#### Question 5: ####
What is the title of the wine that has the highest points-to-price ratio in the dataset?

In [ ]:
valid = wine[wine['price'].notna() & (wine['price'] > 0)]
valid.loc[(valid['points'] / valid['price']).idxmax(), 'title']

'Bandit NV Merlot (California)'

#### Question 6: ####
Create a series `flavor_counts` that contains two values: the number of wines that has the word "tart" in the `description` column and the number of wines that has the word "berries" in the `description` column. The index of the Series should be "Tart" and "Berries" for the corresponding values.

In [15]:
flavor_counts = pd.Series({
    'Tart': wine['description'].str.contains(r'\btart\b', case=False, na=False).sum(),
    'Berries': wine['description'].str.contains(r'\bberries\b', case=False, na=False).sum()
})
flavor_counts

Tart       3086
Berries    1192
dtype: int64

#### Question 7: ####
Let's convert the points into simple star ratings. A score of 90 or higher counts as 3 stars, a score of at least 80 but less than 90 is 2 stars. Any other score is 1 star.

Also, any wines from France should automatically get 3 stars, regardless of points.

Add this new column `star_ratings` to the dataframe with the number of stars for each wine in the dataset.

In [18]:
# Base stars: 90+ -> 3, 80-89 -> 2, else 1
wine['star_ratings'] = 1
wine.loc[wine['points'] >= 80, 'star_ratings'] = 2
wine.loc[wine['points'] >= 90, 'star_ratings'] = 3
# France gets 3 stars regardless of points
wine.loc[wine['country'] == 'France', 'star_ratings'] = 3

#### Question 8: ####
Who are the most common wine reviewers in the dataset? Create a Series whose index is the taster_twitter_handle category from the dataset, and whose values count how many reviews each person wrote.

In [17]:
wine['taster_twitter_handle'].value_counts()

taster_twitter_handle
@vossroger          13045
@wineschach          7752
@kerinokeefe         5313
@paulgwine           4851
@vboone              4696
@mattkettmann        3035
@JoeCz               2605
@wawinereport        2358
@gordone_cellars     2032
@AnneInVino          1769
@laurbuzz             938
@suskostrzewa         593
@worldwineguys        465
@bkfiona               11
@winewchristina         4
Name: count, dtype: int64

#### Question 9: ####
What combination of countries and varieties are most common? Create a Series whose index is a MultiIndexof {country, variety} pairs. For example, a pinot noir produced in the US should map to {"US", "Pinot Noir"}. Sort the values in the Series in descending order based on wine count.

In [19]:
wine.groupby(['country', 'variety']).size().sort_values(ascending=False)

country  variety                 
US       Pinot Noir                  4918
         Cabernet Sauvignon          3649
         Chardonnay                  3412
France   Bordeaux-style Red Blend    2380
Italy    Red Blend                   1870
                                     ... 
         Torbato                        1
         Vespaiolo                      1
         Vespolina                      1
         Vitovska                       1
Uruguay  Tempranillo-Tannat             1
Length: 1304, dtype: int64

#### Question 10 #####
Create a Series whose index is reviewers and whose values is the average score given out by that reviewer.  
*Hint:* You will need the `taster_name` and `points` columns.

In [20]:
wine.groupby('taster_name')['points'].mean()

taster_name
Alexander Peartree    86.014286
Anna Lee C. Iijima    88.380506
Anne Krebiehl MW      90.587903
Carrie Dykes          86.644444
Christina Pickard     89.500000
Fiona Adams           87.090909
Jeff Jenssen          88.273504
Jim Gordon            88.604331
Joe Czerwinski        88.519770
Kerin O’Keefe         88.827969
Lauren Buzzeo         87.831557
Matt Kettmann         90.021087
Michael Schachner     86.904541
Mike DeSimone         89.030303
Paul Gregutt          89.095032
Roger Voss            88.678957
Sean P. Sullivan      88.666243
Susan Kostrzewa       86.408094
Virginie Boone        89.229557
Name: points, dtype: float64